In [46]:
import langchain

In [47]:
import os
from typing import List, Dict, Any
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

In [48]:
from langchain_core.documents import Document
from langchain_text_splitters import (
	RecursiveCharacterTextSplitter,
	CharacterTextSplitter,
	TokenTextSplitter
)

#### Understanding Document Structure in Langchain

In [49]:
# create a simple document 

doc = Document(
	page_content = "This is the main text content that will be embedded and searched.",
	metadata = {
		"source": "example.txt",
		"page": 1,
		"author": "Dilli Ram",
		"date_created": "2026-8-01",
		"custom_field": "any_value"
	}
)

print("Document structure")
print(f"content: ", doc.page_content)
print(f"content: ", doc.metadata)

Document structure
content:  This is the main text content that will be embedded and searched.
content:  {'source': 'example.txt', 'page': 1, 'author': 'Dilli Ram', 'date_created': '2026-8-01', 'custom_field': 'any_value'}


##### Why metadata matters!

In [50]:
print("\nMetadata is crucial for: ")
print("- Filtering search results.")
print("- Tracking document sources.")
print("- Providing context in responses.")
print("- Debugging and auditing.")


Metadata is crucial for: 
- Filtering search results.
- Tracking document sources.
- Providing context in responses.
- Debugging and auditing.


In [51]:
type(doc)

langchain_core.documents.base.Document

#### Self practice

In [52]:
from langchain_core.documents import Document

doc = Document(
	page_content= """
    University Attendance Policy

    Students must maintain at least 80% attendance.
    Students below 80% attendance may not be allowed to sit for examinations.
    """,
  metadata = {
				"source": "university_policy.txt",
        "department": "Academic",
        "year": 2026
	}
)

print(doc.page_content)
print(doc.metadata)


    University Attendance Policy

    Students must maintain at least 80% attendance.
    Students below 80% attendance may not be allowed to sit for examinations.
    
{'source': 'university_policy.txt', 'department': 'Academic', 'year': 2026}


In [53]:
# multiple documents

from langchain_core.documents import Document

documents = [
  Document(
        page_content="Python is a high-level programming language.",
        metadata={"source": "python.txt", "topic": "programming"}
    ),
  Document(
        page_content="LangChain is a framework for building applications with LLMs.",
        metadata={"source": "langchain.txt", "topic": "AI"}
    ),
  Document(
        page_content="Transformers use self-attention to process sequences.",
        metadata={"source": "transformers.txt", "topic": "deep learning"}
    )
]

In [54]:
print(len(documents))

3


In [55]:
print(documents[0])

page_content='Python is a high-level programming language.' metadata={'source': 'python.txt', 'topic': 'programming'}


In [56]:
print(documents[0].page_content)
print(documents[0].metadata)

Python is a high-level programming language.
{'source': 'python.txt', 'topic': 'programming'}


##### Inspecting document structure

In [57]:
for i, doc in enumerate(documents):
  print(f"Document:", i)
  print(f"Content: ", doc.page_content)
  print(f"Metadata: ", doc.metadata)
  print("-"*75)

Document: 0
Content:  Python is a high-level programming language.
Metadata:  {'source': 'python.txt', 'topic': 'programming'}
---------------------------------------------------------------------------
Document: 1
Content:  LangChain is a framework for building applications with LLMs.
Metadata:  {'source': 'langchain.txt', 'topic': 'AI'}
---------------------------------------------------------------------------
Document: 2
Content:  Transformers use self-attention to process sequences.
Metadata:  {'source': 'transformers.txt', 'topic': 'deep learning'}
---------------------------------------------------------------------------


##### Text files (.txt) - The Simplest Case (#2 -text-files)

In [58]:
import os 

os.makedirs("data/text_files", exist_ok=True)

In [59]:
sample_texts = {
	"data/text_files/python_intro.txt": """
Python programming is an easy-to-learn, high-level computer language used for web apps, data analysis, and automation.Core Python ConceptsVariables: Containers used to store data values in memory.Data Types: Basic classifications for data, such as numbers (integers and floats) and strings (text).Functions: Reusable blocks of code that perform specific tasks, like the built-in print() command.Collections: Structures like lists and dictionaries used to group multiple items together.Loops: Control structures like for and while used to repeat actions multiple times.
""",
	"data/text_files/machine_learning.txt": """
Machine learning (ML) is a branch of artificial intelligence where computers learn patterns from data to make decisions without being explicitly programmed. Instead of writing rigid rules, you train a model on historical data so it can make predictions on new, unseen information.Core Machine Learning ConceptsSupervised Learning: Training a model on labeled data, where the correct answers are already known (e.g., predicting house prices).Unsupervised Learning: Finding hidden patterns or groupings in unlabeled data (e.g., segmenting customers into different buying behaviors).Reinforcement Learning: Teaching an agent to make decisions through a system of rewards and punishments (e.g., training an AI to play chess).
"""
}

In [60]:
for filepath, content in sample_texts.items():
  with open(filepath, 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ sample file created.")

✅ sample file created.


#### TextLoader - Read single file

In [61]:
from langchain_community.document_loaders import TextLoader

# laoding a single file
loader = TextLoader("data/text_files/python_intro.txt", encoding='utf-8')
loader

In [62]:
documents = loader.load()
print(type(documents))
print(documents)

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='\nPython programming is an easy-to-learn, high-level computer language used for web apps, data analysis, and automation.Core Python ConceptsVariables: Containers used to store data values in memory.Data Types: Basic classifications for data, such as numbers (integers and floats) and strings (text).Functions: Reusable blocks of code that perform specific tasks, like the built-in print() command.Collections: Structures like lists and dictionaries used to group multiple items together.Loops: Control structures like for and while used to repeat actions multiple times.\n')]


In [63]:
print(f"Loaded {len(documents)} documents...")
print(f"Document preview: {documents[0].page_content[:100]}")
print(f"Metadata document: {documents[0].metadata}")

Loaded 1 documents...
Document preview: 
Python programming is an easy-to-learn, high-level computer language used for web apps, data analys
Metadata document: {'source': 'data/text_files/python_intro.txt'}


#### Directory Loader - Multiple Text files

In [77]:
from langchain_community.document_loaders import DirectoryLoader

# load all the text files from the directory
dir_loader = DirectoryLoader(
	"data/text_files",
	glob="**/*.txt", 							# pattern to match the files
	loader_kwargs= {'encoding': 'utf-8'},
	show_progress=True
)

documents = dir_loader.load()

print(f"Loaded {len(documents)} documents...")
for i, doc in enumerate(documents):
  print(f"\nDocument {i+1}")
  print(f"source: {doc.metadata["source"]}")
  print(f"Length: {len(doc.page_content)} characters...")

  0%|          | 0/2 [00:00<?, ?it/s]libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
100%|██████████| 2/2 [00:00<00:00, 22.78it/s]

Loaded 2 documents...

Document 1
source: data/text_files/python_intro.txt
Length: 568 characters...

Document 2
source: data/text_files/machine_learning.txt
Length: 721 characters...


#### Loading PDFs

In [89]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/text_files/mypdf.pdf")
documents = loader.load()
print(documents)
print(len(documents))

[Document(metadata={'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'PyPDF', 'creationdate': '2023-08-03T21:32:56+00:00', 'meeting starting date': '19 June 2023', 'moddate': '2023-09-30T08:26:56-04:00', 'ieee article id': '10266218', 'ieee issue id': '10265772', 'subject': '2023 3rd International Conference on Pervasive Computing and Social Networking (ICPCSN);2023; ; ;10.1109/ICPCSN58827.2023.00028', 'ieee publication id': '10265860', 'title': 'An ANPR-Based Automatic Toll Tax Collection System Using Camera', 'meeting ending date': '20 June 2023', 'source': 'data/text_files/mypdf.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content="An ANPR-based Automatic Toll Tax Collection System \nusing Camera \n \nB VeerasekharReddy  \nInformation Technology \nMLR Institute of Technology \nHyderabad, India \nbhargavisekhar68@gmail.com \n \nSahithi Sindhu Gadup

In [90]:
for doc in documents:
  print(doc.metadata)

{'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'PyPDF', 'creationdate': '2023-08-03T21:32:56+00:00', 'meeting starting date': '19 June 2023', 'moddate': '2023-09-30T08:26:56-04:00', 'ieee article id': '10266218', 'ieee issue id': '10265772', 'subject': '2023 3rd International Conference on Pervasive Computing and Social Networking (ICPCSN);2023; ; ;10.1109/ICPCSN58827.2023.00028', 'ieee publication id': '10265860', 'title': 'An ANPR-Based Automatic Toll Tax Collection System Using Camera', 'meeting ending date': '20 June 2023', 'source': 'data/text_files/mypdf.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}
{'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'PyPDF', 'creationdate': '2023-08-03T21:32:56+00:00', 'meeting starting date': '

##### Loading Word documents

In [96]:
import docx

doc = docx.Document()
doc.add_paragraph("I live in Nepal and this country is very beautiful located in the lap of the Himalayas.")
doc.save("data/text_files/mydoc.docx")

In [97]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("data/text_files/mydoc.docx")
documents = loader.load()
print(documents[0].page_content)

I live in Nepal and this country is very beautiful located in the lap of the Himalayas.
